# 실전 프로젝트 1차 eda

- **portfolio**: 프로모션 자체의 속성 테이블
    - offer type
    - reward
    - difficulty
    - duration
    - channels
- **profile**: 고객 프로필 테이블
    - gender
    - age
    - income
    - became_member_on
- **transcript**: 고객 행동 로그 테이블
    - 이벤트 종류
    - 이벤트 시점
    - 결제 금액 / 오퍼 ID 등 세부 값

In [5]:
import numpy as np
import pandas as pd
from IPython.display import display
import warnings
import platform
import matplotlib.pyplot as plt

# 출력 설정
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 참고: seed 고정으로 팀원 간 동일한 결과 재현 가능
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


## 데이터 로드

In [27]:
# ============================================================
# 1. 원본 데이터 로드
# ============================================================
df_portfolio = pd.read_csv("data/portfolio.csv", index_col=0)
df_profile = pd.read_csv("data/profile.csv", index_col=0)
df_transcript = pd.read_csv("data/transcript.csv", index_col=0)

In [28]:
# ============================================================
# 2. 데이터 기본 구조 확인
# ============================================================
print("===== 기본 크기 확인 =====")
print("\n[portfolio]")
print("shape :", df_portfolio.shape)
display(df_portfolio.head())
print()

print("[profile]")
print("shape :", df_profile.shape)
display(df_profile.head())
print()

print("[transcript]")
print("shape :", df_transcript.shape)
display(df_transcript.head())
print()


===== 기본 크기 확인 =====

[portfolio]
shape : (10, 6)


,reward,channels,difficulty,duration,offer_type,id
0,10,"['email', 'mobile', 'social']",10,7,bogo,ae264e3637204a6fb9bb56bc8210ddfd
1,10,"['web', 'email', 'mobile', 'social']",10,5,bogo,4d5c57ea9a6940dd891ad53e9dbe8da0
2,0,"['web', 'email', 'mobile']",0,4,informational,3f207df678b143eea3cee63160fa8bed
3,5,"['web', 'email', 'mobile']",5,7,bogo,9b98b8c7a33c4b65b9aebfe6a799e6d9
4,5,"['web', 'email']",20,10,discount,0b1e1539f2cc45b7b9fa7c272da2e1d7



[profile]
shape : (17000, 5)


,gender,age,id,became_member_on,income
0,NaN,118,68be06ca386d4c31939f3a4f0e3dd783,20170212,NaN
1,F,55,0610b486422d4921ae7d2bf64640c50b,20170715,112000.0
2,NaN,118,38fe809add3b4fcf9315a9694bb96ff5,20180712,NaN
3,F,75,78afa995795e4d85b5d9ceeca43f5fef,20170509,100000.0
4,NaN,118,a03223e636434f42ac4c3df47e8bac43,20170804,NaN



[transcript]
shape : (306534, 4)


,person,event,value,time
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,{'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'},0
1,a03223e636434f42ac4c3df47e8bac43,offer received,{'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'},0
2,e2127556f4f64592b11af22de27a7932,offer received,{'offer id': '2906b810c7d4411798c6938adc9daaa5'},0
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,{'offer id': 'fafdcd668e3743c1bb461111dcafc2a4'},0
4,68617ca6246f4fbc85e91a2a49552598,offer received,{'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'},0


portfolio: 10행 × 6열\
profile: 17,000행 × 5열\
transcript: 306,534행 × 4열

1. portfolio: 프로모션의 조건과 유형을 담은 테이블
    - reward : 프로모션 완료 시 고객에게 주는 보상 금액
    - channels : 프로모션이 전달된 채널 목록(web, email, mobile, social)
    - difficulty : 보상을 받기 위해 필요한 최소 지출 금액
    - duration : 프로모션 유효 기간(일 단위)
    - offer_type : 프로모션 유형(bogo, discount, informational)
    - id : 프로모션 고유 ID

2. profile: 고객의 인구통계 및 가입 정보를 담은 테이블
    - gender : 고객 성별
    - age : 고객 나이
    - id : 고객 고유 ID
    - became_member_on : 멤버십 가입일(YYYYMMDD 형식)
    - income : 고객 연간 소득(달러)
    
3. transcript: 고객의 프로모션 반응과 거래 행동을 시간 순으로 기록한 로그 테이블
    - person : 행동을 한 고객 ID
    - event : 행동 유형(transaction, offer received, offer viewed, offer completed)
    - value : 이벤트별 상세 정보가 들어 있는 딕셔너리형 문자열
    - time : 이벤트 발생 시점(상대 시간)


In [29]:
# ============================================================
# 3. 컬럼 정보 / 결측치 확인
# ============================================================
def check_basic_info(df, df_name, exclude_cols=None):
    
    print(f"\n{'='*80}")
    print(f"{df_name}의 컬럼 정보 / 결측치 확인 정보 요약")
    print(f"{'='*80}\n")
    
    # 1. 전체 요약
    overview_df = pd.DataFrame({
        '항목': ['행 개수', '열 개수', '중복 행 개수'],
        '값': [df.shape[0], df.shape[1], df.duplicated().sum()]
    })
    
    # 2. 컬럼별 요약
    summary_df = pd.DataFrame({
        '데이터타입': df.dtypes.astype(str),
        '결측치 개수': df.isnull().sum(),
        '결측치 비율(%)': (df.isnull().sum() / len(df) * 100).round(2),
        '고유값 개수': df.nunique(dropna=True)
    })
    
    # 3. 보기 좋게 정렬
    summary_df = summary_df.sort_values(
        by=['결측치 개수', '고유값 개수'],
        ascending=[False, False]
    )
    
    print("[전체 요약]")
    display(overview_df)
    
    print("[컬럼별 요약]")
    display(summary_df)

In [30]:
check_basic_info(df_portfolio, "portfolio")
check_basic_info(df_profile, "profile")
check_basic_info(df_transcript, "transcript")


portfolio의 컬럼 정보 / 결측치 확인 정보 요약

[전체 요약]


,항목,값
0,행 개수,10
1,열 개수,6
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,결측치 개수,결측치 비율(%),고유값 개수
id,str,0,0.0,10
reward,int64,0,0.0,5
difficulty,int64,0,0.0,5
duration,int64,0,0.0,5
channels,str,0,0.0,4
offer_type,str,0,0.0,3



profile의 컬럼 정보 / 결측치 확인 정보 요약

[전체 요약]


...

,데이터타입,결측치 개수,결측치 비율(%),고유값 개수
income,float64,2175,12.79,91
gender,str,2175,12.79,3
id,str,0,0.00,17000
became_member_on,int64,0,0.00,1716
age,int64,0,0.00,85



transcript의 컬럼 정보 / 결측치 확인 정보 요약

[전체 요약]


,항목,값
0,행 개수,306534
1,열 개수,4
2,중복 행 개수,397


[컬럼별 요약]


,데이터타입,결측치 개수,결측치 비율(%),고유값 개수
person,str,0,0.0,17000
value,str,0,0.0,5121
time,int64,0,0.0,120
event,str,0,0.0,4


In [31]:
def create_statistics_summary(df, df_name, exclude_cols=None):
    
    print(f"\n{'='*80}")
    print(f"{df_name} 관련 데이터 기초통계량")
    print(f"{'='*80}\n")
    
    # 1-1. exclude_cols에 속한 컬럼 제외한 데이터프레임 생성
    df_copied = df.copy()
    if exclude_cols:
        df_copied = df_copied.drop(columns=exclude_cols, errors='ignore')
    
    # 1-2. 기초 통계량
    stats_df = df_copied.describe().T
    
    # 1-4. 컬럼명 한글로 변경
    stats_df.rename(columns={
        'count': '개수', # 결측치가 아닌 값의 개수
        'mean': '평균',
        'std': '표준편차',
        'min': '최솟값',
        '25%': 'Q1의 경계값',
        '50%': '중앙값',
        '75%': 'Q3의 경계값',
        'max': '최댓값',
    }, inplace=True)
    
    display(stats_df)
    
    return stats_df

In [32]:
stats_df_process = create_statistics_summary(df_portfolio, 'df_portfolio')
stats_df_sensor = create_statistics_summary(df_profile, 'df_profile')
stats_df_sensor = create_statistics_summary(df_transcript, 'df_transcript')


df_portfolio 관련 데이터 기초통계량



,개수,평균,표준편차,최솟값,Q1의 경계값,중앙값,Q3의 경계값,최댓값
reward,10.0,4.2,3.583915,0.0,2.0,4.0,5.0,10.0
difficulty,10.0,7.7,5.831905,0.0,5.0,8.5,10.0,20.0
duration,10.0,6.5,2.321398,3.0,5.0,7.0,7.0,10.0



df_profile 관련 데이터 기초통계량



,개수,평균,표준편차,최솟값,Q1의 경계값,중앙값,Q3의 경계값,최댓값
age,17000.0,6.253141e+01,26.738580,18.0,45.0,58.0,73.0,118.0
became_member_on,17000.0,2.016703e+07,11677.499961,20130729.0,20160526.0,20170802.0,20171230.0,20180726.0
income,14825.0,6.540499e+04,21598.299410,30000.0,49000.0,64000.0,80000.0,120000.0



df_transcript 관련 데이터 기초통계량



,개수,평균,표준편차,최솟값,Q1의 경계값,중앙값,Q3의 경계값,최댓값
time,306534.0,366.38294,200.326314,0.0,186.0,408.0,528.0,714.0


profile에

gender, income 결측치: 2,175건\
age = 118인 값이 존재


<bound method NDFrame.describe of    reward                              channels  difficulty  duration  \
0      10         ['email', 'mobile', 'social']          10         7   
1      10  ['web', 'email', 'mobile', 'social']          10         5   
2       0            ['web', 'email', 'mobile']           0         4   
3       5            ['web', 'email', 'mobile']           5         7   
4       5                      ['web', 'email']          20        10   
5       3  ['web', 'email', 'mobile', 'social']           7         7   
6       2  ['web', 'email', 'mobile', 'social']          10        10   
7       0         ['email', 'mobile', 'social']           0         3   
8       5  ['web', 'email', 'mobile', 'social']           5         5   
9       2            ['web', 'email', 'mobile']          10         7   

      offer_type                                id  
0           bogo  ae264e3637204a6fb9bb56bc8210ddfd  
1           bogo  4d5c57ea9a6940dd891ad53e9dbe8da0  
2  